# IPL Clean Analysis and Multi-Model Training

This notebook loads your IPL data, cleans it, performs quick EDA, and trains several models for regression and classification.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, f1_score, roc_auc_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier, GradientBoostingClassifier

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)

In [ ]:
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
candidates = [ROOT / 'ipl_colab.csv', ROOT / 'data' / 'processed' / 'ipl_features.csv']
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('Expected ipl_colab.csv or data/processed/ipl_features.csv')

df_raw = pd.read_csv(data_path, low_memory=False)
print('Loaded:', data_path)
print('Shape:', df_raw.shape)
df_raw.head()

In [ ]:
def clean_dataframe(df):
    out = df.copy().drop_duplicates()
    for c in out.columns:
        if 'date' in c.lower():
            dt = pd.to_datetime(out[c], errors='coerce')
            if dt.notna().sum() > 0:
                out[c] = dt
    for c in out.select_dtypes(include=['object']).columns:
        out[c] = out[c].astype(str).str.strip()
        out[c] = out[c].replace({'': np.nan, 'nan': np.nan, 'None': np.nan})
    return out

df = clean_dataframe(df_raw)
print('Raw shape:', df_raw.shape)
print('Clean shape:', df.shape)
df.head()

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
display(missing.to_frame('missing_count').head(20))

num_cols = df.select_dtypes(include=['number']).columns.tolist()
if num_cols:
    display(df[num_cols].describe().T.head(20))
    plt.figure(figsize=(8,4))
    df[num_cols[0]].dropna().hist(bins=30)
    plt.title(f'Distribution of {num_cols[0]}')
    plt.show()

In [ ]:
def build_preprocessor(X):
    num = X.select_dtypes(include=['number', 'bool', 'datetime64[ns]']).columns.tolist()
    cat = [c for c in X.columns if c not in num]
    return ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat),
    ])

def run_regression(frame, target):
    d = frame.dropna(subset=[target]).copy()
    X, y = d.drop(columns=[target]), d[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    prep = build_preprocessor(X_train)
    models = {
        'LinearRegression': LinearRegression(),
        'RandomForestRegressor': RandomForestRegressor(n_estimators=250, random_state=42, n_jobs=-1),
        'GradientBoostingRegressor': GradientBoostingRegressor(random_state=42),
    }
    rows = []
    for n, m in models.items():
        p = Pipeline([('prep', prep), ('model', m)])
        p.fit(X_train, y_train)
        pred = p.predict(X_test)
        rows.append({'Model': n, 'MAE': float(mean_absolute_error(y_test, pred)), 'RMSE': float(mean_squared_error(y_test, pred)**0.5), 'R2': float(r2_score(y_test, pred))})
    return pd.DataFrame(rows).sort_values('RMSE').reset_index(drop=True)

def run_classification(frame, target):
    d = frame.dropna(subset=[target]).copy()
    X, y = d.drop(columns=[target]), d[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y if y.nunique()>1 else None)
    prep = build_preprocessor(X_train)
    models = {
        'LogisticRegression': LogisticRegression(max_iter=1000),
        'RandomForestClassifier': RandomForestClassifier(n_estimators=250, random_state=42, n_jobs=-1),
        'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42),
    }
    rows = []
    binary = y.nunique() == 2
    for n, m in models.items():
        p = Pipeline([('prep', prep), ('model', m)])
        p.fit(X_train, y_train)
        pred = p.predict(X_test)
        row = {'Model': n, 'Accuracy': float(accuracy_score(y_test, pred)), 'F1_weighted': float(f1_score(y_test, pred, average='weighted'))}
        if binary and hasattr(p.named_steps['model'], 'predict_proba'):
            prob = p.predict_proba(X_test)[:, 1]
            row['ROC_AUC'] = float(roc_auc_score(y_test, prob))
        rows.append(row)
    return pd.DataFrame(rows).sort_values('Accuracy', ascending=False).reset_index(drop=True)

In [ ]:
reg_target = 'total' if 'total' in df.columns else ('total_runs' if 'total_runs' in df.columns else None)
if reg_target:
    reg_results = run_regression(df, reg_target)
    display(reg_results)
    print('Best regression model:', reg_results.iloc[0]['Model'])
else:
    print('Regression skipped: no total/total_runs column found.')

In [ ]:
if 'win' in df.columns:
    clf_df = df.copy()
    clf_target = 'win'
else:
    clf_df = df.copy()
    if reg_target is None:
        raise ValueError('Classification skipped: no win column and no numeric target for derived label.')
    threshold = clf_df[reg_target].median()
    clf_df['high_score_label'] = (clf_df[reg_target] >= threshold).astype(int)
    clf_target = 'high_score_label'

clf_results = run_classification(clf_df, clf_target)
display(clf_results)
print('Best classification model:', clf_results.iloc[0]['Model'])